# 02. Pipeline Modular de Pré-processamento de Texto

**Contexto:** Tech Challenge 04 - Voice of Customer Intelligence com RAG  
**Fundamentação Teórica:** Fase 4 | Aula 02 - *Pré-processamento de Texto*  

A máxima de NLP é simples: **"Garbage in, garbage out"**. Cerca de 80% do esforço em projetos reais de linguagem natural é dedicado à coleta, limpeza e estruturação dos dados.

Neste notebook, construímos um pipeline profissional, modular e idempotente para o português brasileiro, incorporando as três etapas canônicas:
1. **Limpeza e Normalização com Regex:** remoção de ruídos (HTML, URLs) e expansão de contrações informais de PT-BR (`CONTRACTIONS`).
2. **Tokenização:** quebra do texto em unidades significativas.
3. **Redução Morfológica e Stop Words:** teste do stemmer RSLP e filtragem consciente de stop words mantendo a integridade de termos de negação.
4. **Integração Modular:** aplicação da classe `PipelinePreprocessamento` desenvolvida em `src/preprocessing.py` e enriquecimento relacional com metadados de pedidos e produtos.

In [1]:
import sys
import os
sys.path.append('../')

import pandas as pd
import numpy as np
import re
from src.preprocessing import (
    CONTRACTIONS,
    limpar_normalizar,
    expandir_contracoes,
    tokenizar,
    obter_stopwords_pt,
    aplicar_stemming_rslp,
    PipelinePreprocessamento,
)

DATA_DIR = '../data/'
PROCESSED_DIR = '../data/processed/'
os.makedirs(PROCESSED_DIR, exist_ok=True)

## 1. O Caos do Texto Bruto e a Expansão de Contrações (PT-BR)
Em avaliações de e-commerce brasileiro, contrações como *"vc"*, *"pq"*, *"tbm"*, *"obg"*, *"n"* são extremamente comuns. Se não forem tratadas, cada uma gera tokens isolados que fragmentam a semântica.

In [2]:
exemplo_sujo = "<p>Adorei o produto!! Entrega super rápida https://rastreio.com/123 VC PRECISAVA VER TBM, mt obg!!!</p>"
exemplo_limpo = limpar_normalizar(exemplo_sujo)

print("Texto Bruto Original:")
print(exemplo_sujo)
print("\nTexto Limpo e Normalizado pelo Pipeline:")
print(exemplo_limpo)

Texto Bruto Original:
<p>Adorei o produto!! Entrega super rápida https://rastreio.com/123 VC PRECISAVA VER TBM, mt obg!!!</p>

Texto Limpo e Normalizado pelo Pipeline:
adorei o produto!! entrega super rápida você precisava ver tbm, muito obg!!!


## 2. A Armadilha das Stop Words em Análise de Sentimento (Aula 02)
Conforme enfatizado na Aula 02: **a sensibilidade ao downstream define o que é um bom pré-processamento**.  
Se usarmos uma lista ingênua de stop words, palavras de negação como **"não"** e **"nunca"** são descartadas. Observe a catástrofe resultante:

In [3]:
review_negativa = "Não recebi o produto e nunca fui tão mal atendido"

# Stopwords padrão (sem proteção)
stopwords_brutas = obter_stopwords_pt(preservar_negacoes=False)
tokens_sem_protecao = [t for t in review_negativa.lower().split() if t not in stopwords_brutas]

# Stopwords com salvaguarda profissional (Aula 02)
stopwords_protegidas = obter_stopwords_pt(preservar_negacoes=True)
tokens_protegidos = [t for t in review_negativa.lower().split() if t not in stopwords_protegidas]

print(f"Frase Original: '{review_negativa}'")
print(f"Stopwords ingênuas (INVERTE O SENTIMENTO): {' '.join(tokens_sem_protecao)}")
print(f"Stopwords com preservação de negações: {' '.join(tokens_protegidos)}")

Frase Original: 'Não recebi o produto e nunca fui tão mal atendido'
Stopwords ingênuas (INVERTE O SENTIMENTO): recebi produto nunca fui tão mal atendido
Stopwords com preservação de negações: não recebi produto nunca fui tão mal atendido


## 3. Redução Morfológica em Português: Stemming RSLP (Aula 02)
O português brasileiro possui mais de 50 formas flexionadas por verbo. Para métodos clássicos de representação (BoW e TF-IDF), agrupar essas variações pelo radical reduz o vocabulário e combate a esparsidade.

In [4]:
palavras_teste = ["entregou", "entregar", "entregaram", "entrega", "maravilhoso", "felicidades"]
radicais = aplicar_stemming_rslp(palavras_teste)

for p, r in zip(palavras_teste, radicais):
    print(f"{p:<15} -> {r}")

entregou        -> entregou
entregar        -> entregar
entregaram      -> entregaram
entrega         -> entrega
maravilhoso     -> maravilhoso
felicidades     -> felicidades


## 4. Aplicação do Pipeline no Corpus do Olist
Agora aplicamos a classe `PipelinePreprocessamento` desenvolvida em `src/preprocessing.py` sobre toda a base de comentários do Olist.

In [5]:
df_reviews = pd.read_csv(f"{DATA_DIR}olist_order_reviews_dataset.csv")
df_reviews = df_reviews.dropna(subset=['review_comment_message']).copy()

# Instanciamos o pipeline configurado para RAG e VoC
# (mantemos pontuação expressiva e preservamos negações semânticas)
pipeline = PipelinePreprocessamento(
    remove_acentos=False,
    lowercase=True,
    manter_pontuacao=True,
    remover_stopwords=False, # Para RAG com LLM mantemos a prosa natural
    preservar_negacoes=True
)

df_reviews['clean_comment'] = df_reviews['review_comment_message'].apply(pipeline.processar_texto)
df_reviews['review_comment_title'] = df_reviews['review_comment_title'].fillna('').astype(str).apply(pipeline.processar_texto)

print(f"Total de avaliações processadas: {len(df_reviews):,}")
df_reviews[['review_id', 'review_score', 'clean_comment']].head()

Total de avaliações processadas: 40,977


,review_id,review_score,clean_comment
3,e64fb393e7b32834bb789ff8bb30750e,5,recebi bem antes do prazo estipulado.
4,f7c4243c7fe1938f181bec41a392bdeb,5,parabéns lojas lannister adorei comprar pela i...
9,8670d52e15e00043ae7de4c01cc2fe06,4,aparelho eficiente. no site a marca do aparelh...
12,4b49719c8a200003f700d3d986ea1a19,4,"mas um pouco ,travando...pelo valor está boa."
15,3948b09f7c818e2d86c9a546758b2335,5,"vendedor confiável, produto ok e entrega antes..."


## 5. Enriquecimento de Dados com Metadados (Desafio Adicional)
Para capacitar o RAG a responder perguntas contextuais (ex: "Quais os problemas do setor de brinquedos?" ou "Como foram as entregas em 2017?"), realizamos a consolidação das tabelas dimensionais.

In [6]:
df_orders = pd.read_csv(f"{DATA_DIR}olist_orders_dataset.csv")
df_items = pd.read_csv(f"{DATA_DIR}olist_order_items_dataset.csv")
df_products = pd.read_csv(f"{DATA_DIR}olist_products_dataset.csv")

# Join 1: Pedidos (data da compra e data de entrega)
df_merged = df_reviews.merge(
    df_orders[['order_id', 'order_status', 'order_purchase_timestamp', 'order_delivered_customer_date']],
    on='order_id',
    how='left'
)

# Join 2: Itens do pedido
df_merged = df_merged.merge(
    df_items[['order_id', 'product_id', 'price']],
    on='order_id',
    how='left'
)

# Join 3: Categoria do produto
df_merged = df_merged.merge(
    df_products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)

# Tratamento de nulos em categoria decorrentes do join
df_merged['product_category_name'] = df_merged['product_category_name'].fillna('desconhecida')

# Agregação em nível de review_id para evitar duplicações de pedidos multi-itens
df_final = df_merged.groupby('review_id').agg({
    'review_score': 'first',
    'review_comment_title': 'first',
    'clean_comment': 'first',
    'order_purchase_timestamp': 'first',
    'order_delivered_customer_date': 'first',
    'product_category_name': lambda x: ", ".join(list(set([str(i) for i in x if str(i) != 'nan'])))
}).reset_index()

# --- Metadados estruturados para filtragem no banco vetorial (Desafio Adicional) ---
# Categoria principal (primeira categoria do pedido) como campo escalar -> filtro exato / por lista.
df_final['categoria'] = (
    df_final['product_category_name']
    .str.split(',').str[0].str.strip()
    .replace('', 'desconhecida')
)
# Data da compra decomposta em campos NUMÉRICOS: o ChromaDB só faz range ($gte/$lte) em números,
# strings aceitam apenas $eq/$in. ano_mes (YYYYMM) habilita filtro por período contínuo.
dt = pd.to_datetime(df_final['order_purchase_timestamp'], errors='coerce')
df_final['ano'] = dt.dt.year.fillna(0).astype(int)
df_final['mes'] = dt.dt.month.fillna(0).astype(int)
df_final['ano_mes'] = (df_final['ano'] * 100 + df_final['mes']).astype(int)  # ex.: 201704

# Criação do documento canônico contextualizado para o RAG
def formatar_documento_rag(row):
    titulo = f" Título: {row['review_comment_title']} |" if row['review_comment_title'] else ""
    return f"Categoria: {row['product_category_name']} | Nota: {row['review_score']} | Data: {str(row['order_purchase_timestamp'])[:10]} |{titulo} Comentário: {row['clean_comment']}"

df_final['rag_document'] = df_final.apply(formatar_documento_rag, axis=1)

# Exportação dos dados processados (inclui explicitamente os metadados estruturados)
colunas_export = ['review_id', 'rag_document', 'review_score', 'categoria', 'ano', 'mes', 'ano_mes']
output_path = f"{PROCESSED_DIR}reviews_enriched_for_rag.csv"
df_final[colunas_export].to_csv(output_path, index=False)
print(f"\nCorpus consolidado e enriquecido salvo com sucesso em: {output_path}")
print(f"Total de documentos únicos: {len(df_final):,}")
print(f"Metadados exportados: {colunas_export}")
df_final[colunas_export].head()


Corpus consolidado e enriquecido salvo com sucesso em: ../data/processed/reviews_enriched_for_rag.csv
Total de documentos únicos: 40,668
Metadados exportados: ['review_id', 'rag_document', 'review_score', 'categoria', 'ano', 'mes', 'ano_mes']


,review_id,rag_document,review_score,categoria,ano,mes,ano_mes
0,00020c7512a52e92212f12d3e37513c0,Categoria: casa_construcao | Nota: 5 | Data: 2...,5,casa_construcao,2018,4,201804
1,00046a69550325aea5fb89f65c7387f2,Categoria: telefonia | Nota: 5 | Data: 2018-06...,5,telefonia,2018,6,201806
2,0005534973388c830bb858cfba83b17b,Categoria: esporte_lazer | Nota: 5 | Data: 201...,5,esporte_lazer,2017,8,201708
3,00055e36e9608fe969231e551983a69c,Categoria: brinquedos | Nota: 5 | Data: 2017-1...,5,brinquedos,2017,11,201711
4,0005949d4c047d64863a6874338139ba,Categoria: utilidades_domesticas | Nota: 5 | D...,5,utilidades_domesticas,2017,6,201706
